---
title: Глава 9. Подзапросы
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-09-21
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Инструмент сборки статических сайтов
    JupySQL: Расширение для запуска и подсветки SQL в Jupyter
    GitHub: Платформа хостинга репозиториев и совместной разработки
    GitHub Pages: Сервис бесплатного хостинга статических сайтов
    GitHub Actions: Платформа автоматизации рабочих процессов и CI/CD
    Pandas: Библиотека Python для анализа и обработки данных
    Polars: Мощный аналог Pandas на Rust/Python
---

In [1]:
import pandas as pd
import sql
import sqlalchemy as sa

# Настройка вывода Pandas
pd.set_option('display.max_rows', 20)

# Подключение и настройка SQL-магии
%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.autopandas = True

# Подключение к базе данных
connection_url = sa.engine.URL.create(
    drivername='mysql+pymysql',
    host='localhost',
    port=3306,
    database='sakila',
    username='root',
    password='*UHB5rdx',
)
engine = sa.create_engine(connection_url)

%sql engine

# Статус инициализации ячейки
print(f"Pandas ver. {pd.__version__}: порог усечения строк уменьшен до 20")
print(f"SQLAlchemy ver. {sa.__version__}: подключение создано")
print(f"JupySQL ver. {sql.__version__}: подключен через SQLAlchemy Engine")

Pandas ver. 3.0.5: порог усечения строк уменьшен до 20
SQLAlchemy ver. 2.0.52: подключение создано
JupySQL ver. 0.11.1: подключен через SQLAlchemy Engine


## Что такое подзапрос

*Подзапрос* – это запрос, содержащийся в другой инструкции SQL *(содержащей инструкции)*.

Подзапрос всегда заключен в круглые скобки и обычно выполняется перед содержащей инструкцией. Подобно любому запросу, подзапрос возвращает результирующий набор, который может состоять из:
- одной строки с одним столбцом;
- нескольких строк с одним столбцом;
- нескольких строк с несколькими столбцами.

Тип результирующего набора, возвращаемого подзапросом, определяет как можно его использовать и какие операторы может использовать содержащая инструкция для взаимодействия с данными, возвращаемыми подзапросом.

Когда содержащая инструкция завершает выполнение, данные, возвращенные любыми подзапросами уничтожаются, что делает подзапрос действующим как вр*е*менная таблица с *областью видимости инструкции* (это означает, что сервер освобождает всю память, выделенную для результатов подзапроса после выполнения инструкции SQL).

Мы уже видели примеры подзапросов в предыдущих главах, вот еще один простой пример для начала:

In [2]:
%%sql
SELECT customer_id, first_name, last_name
FROM customer
WHERE customer_id = (
    SELECT MAX(customer_id)
    FROM customer
);

1 rows affected.

,customer_id,first_name,last_name
0,599,AUSTIN,CINTRON


В этом примере подзапрос возвращает максимальное значение, найденное в столбце *customer_id* в таблице *customer*, а затем содержащая инструкция возвращает данные об этом клиенте.

Если не понимаете что делает подзапрос, можно запустить его сам по себе (без скобок), чтобы увидеть что он возвращает:

In [3]:
%%sql
SELECT MAX(customer_id)
FROM customer;

1 rows affected.

,MAX(customer_id)
0,599


Этот подзапрос вернул одну строку с одним столбцом, что позволяет использовать его как одно из выражений равенства.

В принципе мы можем взять значение, возвращенное подзапросом и заменить им наш подзапрос в содержащем запросе:

In [4]:
%%sql
SELECT customer_id, first_name, last_name
FROM customer
WHERE customer_id = 599;

1 rows affected.

,customer_id,first_name,last_name
0,599,AUSTIN,CINTRON


В рассмотренном случае *подзапрос* полезен, поскольку позволяет получить информацию о клиенте с наибольшем идентификатором в **одном запросе** вместо поиска максимального значения customer_id c помощью одного запроса, а затем написания второго запроса для получения требуемых данных из таблицы customer.

## Типы подзапросов

Наряду с отмеченными ранее различиями в отношении типа результурующего набора, возвращаемого подзапросом (одна строка / столбец; одна строка / несколько столбцов; несколько строк / столбцов) можно использовать для классификации подзапросов другую характеристику.
- Одни поздапросы полностью автономные (именуются *некоррелированными подзапросами*),
- В то время как другие ссылаются на столбцы из содержащей инструкции (именуются *корреллрованными подзапросами*).

## Некоррелированные подзапросы

Пример, показанный выше в этой главе – *некоррелированный подзапрос*, который может быть выполнен отдельно и не ссылается ни на что из содержащей инструкции.

Большинство подзапросов, с которыми вы столкнетесь, будут принадлежать этому типу. Помимо того что это некоррелируемый подзапрос, данный пример также возвращает результирующий набор, содержащий только одну строку и один столбец. Этот тип подзапроса известен как *скалярный подзапрос* и может появляться на любой стороне условия, использующего обычные операторы сравнения (=, <>, <, >, <=, >=).

В следующем примере показано как можно использовать *скалярный подзапрос* в условии неравенства:

In [5]:
%%sql
SELECT city_id, city
FROM city
WHERE country_id <> (
    SELECT country_id
    FROM country
    WHERE country = 'India'
);

540 rows affected.

,city_id,city
0,1,A Coruña (La Coruña)
1,2,Abha
2,3,Abu Dhabi
3,4,Acuña
4,5,Adana
...,...,...
535,596,Zaria
536,597,Zeleznogorsk
537,598,Zhezqazghan
538,599,Zhoushan


Этот запрос возвращает все города, находящиеся не в Индии. Подзапрос возвращает идентификатор страны для Индии, а содержащий его запрос возвращает все города, идентификатор страны которых не равен полученному.

Хотя подзапрос в этом примере довольно прост, фактически подзапросы могут быть настолько сложными, насколько это нужно, и могут использовать любые доступные предложения запроса (select, from, where, group by, having и order by).

Если подзапрос используется в условии равенства, но возвращает более одной строки, то будет получено сообщение об ошибке. Например, если изменим предыдущий запрос так, чтобы подзапрос возвращал *все* страны, *за исключением* Индии, то получим сообщение об ошибке:

In [8]:
%%sql
SELECT city_id, city
FROM city
WHERE country_id <> (
    SELECT country_id
    FROM country
    WHERE country <> 'India'
);

RuntimeError: (pymysql.err.OperationalError) (1242, 'Subquery returns more than 1 row')
[SQL: SELECT city_id, city
FROM city
WHERE country_id <> (
    SELECT country_id
    FROM country
    WHERE country <> 'India'
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


Если выполните подзапрос отдельно, то увидете что он возвращает больше одной строки:

In [9]:
%%sql
SELECT country_id
FROM country
WHERE country <> 'India';

108 rows affected.

,country_id
0,1
1,2
2,3
3,4
4,5
...,...
103,105
104,106
105,107
106,108


Содержащий запрос не выполняется, потому что выражение (country_id) не может быть приравнено к набору выражений (1, 2, 3,... 109). Другими словами, нельзя приравнять одну вещь и набор вещей.

### Подзапросы с несколькими строками и одним столбцом

Если подзапрос возвращает более одной строки, его нельзя использовать в условии равенства (как показано в предыдущем примере).

Однако, есть четыре дополнительных оператора, которые можно использовать для создания условий с этими типами подзапросов.

---

#### Операторы in и not in

Хотя *проверить на равенство* одно значение с набором значений нельзя, можно проверить входит ли конкретное значение в набор.

Следующий пример (пусть и не использующий подзапрос) демонстрирует как создать условие, которое использует оператор `in` для поиска для значения в наборе значений:

In [14]:
%%sql
SELECT country_id
FROM country
WHERE country IN ('Canada', 'Mexico');

2 rows affected.

,country_id
0,20
1,60


Выражение в левой части условия – столбец country, а в правой – набор строк. Оператор `in` проверяет, входит ли строка из столбца country в этот набор; если да, то условие выполняется и строка добавляется к результирующему набору.

Тот же результат можно получить, использую два условия равенства, например:

In [11]:
%%sql
SELECT country_id
FROM country
WHERE country = 'Canada' OR country = 'Mexico';

2 rows affected.

,country_id
0,20
1,60


Хотя такой подход кажется разумным (когда набор содержит только два выражения), легко понять почему одно условие с использованием оператора `in` предпочтительнее, когда множество содержит десятки (или сотни, тысячи и т.д.) значений.

Хотя иногда набор строк, дат или чисел для использования с одной стороны условия создается вручную, куда чаще такой набор создается с помощью подзапроса, который возвращает одну или несколько строк.

В следующем запросе оператор `in` используется с подзапросом в правой части условия фильтра, чтобы получить все города, которые находятся в Канаде или Мексике:

In [15]:
%%sql
SELECT city_id, city
FROM city
WHERE country_id IN (
    SELECT country_id
    FROM country
    WHERE country IN ('Canada', 'Mexico')
);

37 rows affected.

,city_id,city
0,179,Gatineau
1,196,Halifax
2,300,Lethbridge
3,313,London
4,383,Oshawa
...,...,...
32,452,San Juan Bautista Tuxtepec
33,541,Torreón
34,556,Uruapan
35,563,Valle de Santiago


Помимо проверки имеется ли некоторое значение в наборе, мы можем проверить обратное используя оператор `not in`. Вот версия предыдущего запроса, использующая `not in` вместо `in`:

In [16]:
%%sql
SELECT city_id, city
FROM city
WHERE country_id NOT IN (
    SELECT country_id
    FROM country
    WHERE country IN ('Canada', 'Mexico')
);

563 rows affected.

,city_id,city
0,1,A Coruña (La Coruña)
1,2,Abha
2,3,Abu Dhabi
3,5,Adana
4,6,Addis Abeba
...,...,...
558,596,Zaria
559,597,Zeleznogorsk
560,598,Zhezqazghan
561,599,Zhoushan


Этот запрос нахоит все города, расположенные *не* в Канаде или Мексике.

---

#### Оператор all

В то время как оператор `in` используется чтобы выяснить имеется ли выражение в наборе выражений, оператор `all` позволяет сравнивать отдельное значение и каждое значение в наборе.

Чтобы создать такое условие, нужно использовать один из операторов сравнения (=, <>, <, > и т.д.) в сочетании с оператором `all`. Например, следующий запрос находит всех клиентов, которые никогда не получали фильмы напрокат бесплатно:

In [17]:
%%sql
SELECT first_name, last_name
FROM customer
WHERE customer_id <> ALL (
    SELECT customer_id
    FROM payment
    WHERE amount = 0
);

576 rows affected.

,first_name,last_name
0,MARY,SMITH
1,PATRICIA,JOHNSON
2,LINDA,WILLIAMS
3,BARBARA,JONES
4,ELIZABETH,BROWN
...,...,...
571,TERRENCE,GUNDERSON
572,ENRIQUE,FORSYTHE
573,FREDDIE,DUGGAN
574,WADE,DELVALLE


Подзапрос возвращает набор идентификаторов клиентов, которые заплатили 0 долларов за прокат фильма, а содержащий запрос возвращает имена всех клиентов, чьи идентификаторы не указаны в результирующем наборе, возвращенном подзапросом.

Если этот подход кажется вам несколько неуклюжим, вы в хорошей компании: большинство людей предпочли бы сформулировать запрос иначе и избежать использования оператора `all`. Для иллюстрации – предыдущий запрос дает те же результаты, что и следующий пример, в котором используется оператор `not in`:

In [ ]:
%%sql
SELECT first_name, last_name
FROM customer
WHERE customer_id NOT IN (
    SELECT customer_id
    FROM payment
    WHERE amount = 0
);

Выбор того или иного запроса – вопрос вкуса, но я думаю, что большинство людей сочтут версию, которая использует `not in` более легкой для понимания.

:::{tip} При использовании `not in` или `<> all`
Для сравнения значения с набором значений нужно быть осторожным и убедиться, что набор значений не содержит `null`, потому что сервер выполняет приравнивание значения с левой стороны выражения к каждому члену набора, а любая попытка приравнять значение к `null` дает `unknown`.

Таким образом, следующий запрос возвращает пустой результирующий набор:
:::

In [24]:
%%sql
SELECT first_name, last_name
FROM customer
WHERE customer_id NOT IN (122, 452, NULL);

,first_name,last_name


:::{topic} *Личная тренировка*
В нашем примере, чтобы защититься от ловушки с `null`, можно внутри подзапроса использовать фильтрацию `is not null`:
```sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.customer_id NOT IN (
    SELECT p.customer_id
    FROM payment p
    WHERE p.amount = 0
      AND p.customer_id IS NOT NULL -- Защита от ловушки с NULL
);
```
:::

Вот еще один пример с использованием оператора `all`, но на этот раз подзапрос находится в предложении `having`:

In [27]:
%%sql
SELECT customer_id, COUNT(*)
FROM rental
GROUP BY customer_id
HAVING COUNT(*) > ALL (
    SELECT COUNT(*)
    FROM rental r
        INNER JOIN customer c ON r.customer_id = c.customer_id
        INNER JOIN address a ON c.address_id = a.address_id
        INNER JOIN city ct ON a.city_id = ct.city_id
        INNER JOIN country co ON ct.country_id = co.country_id
    WHERE co.country IN ('United States', 'Mexico', 'Canada')
    GROUP BY r.customer_id
);

1 rows affected.

,customer_id,COUNT(*)
0,148,46


Подзапрос возвращает общее количество прокатов фильмов для каждого клиента в Северной Америке:

In [33]:
%%sql
SELECT COUNT(*)
FROM rental r
    INNER JOIN customer c ON r.customer_id = c.customer_id
    INNER JOIN address a ON c.address_id = a.address_id
    INNER JOIN city ct ON a.city_id = ct.city_id
    INNER JOIN country co ON ct.country_id = co.country_id
WHERE co.country IN ('United States', 'Mexico', 'Canada')
GROUP BY r.customer_id
ORDER BY COUNT(*) DESC;

71 rows affected.

,COUNT(*)
0,45
1,38
2,35
3,34
4,33
...,...
66,20
67,18
68,18
69,17


А содержащий запрос возвращает всех клиентов, общее количество прокатов фильмов у которых превышает значение у любого из североамериканских клиентов:

In [29]:
%%sql
SELECT customer_id, COUNT(*)
FROM rental
GROUP BY customer_id
HAVING COUNT(*) > 45;

1 rows affected.

,customer_id,COUNT(*)
0,148,46


::::{note} Суть оператора `ALL`
:class: dropdown simple
:open: false
:icon: false

**ALL** – это математическая приставка к любому оператору сравнения (>, <, >=, <=, =, <>), которая превращает условие в требование:
- **Сравнение должно быть верным для каждого элемента списка**.

---
:::{div}
:class: text-sm
Операторы `IN` и `NOT IN` умеют проверять только равенство `=` или неравенство `!=`
:::
```sql
-- Так НЕЛЬЗЯ! Синтаксическая ошибка:
WHERE total_rentals > IN (10, 20, 45)
```
:::{div}
:class: text-sm
А с `ALL` можем использовать знак *больше* или *меньше*:
:::
```sql
-- А вот так МОЖНО:
WHERE total_rentals > ALL (подзапрос)
```

---
:::{div}
:class: text-sm
- `x > ALL (10, 20, 45)` буквально означает: `x > 10 AND x > 20 AND x > 45`. \
То есть **_x_ должен быть строго больше максимума из этого списка** (то есть x > 45).
- `x <> ALL (10, 20, 45)` буквально читается: `x != 10 AND x != 20 AND x != 45` \
То есть **_x_ не совпадает ни с одним элементом множества**, поэтому математически \
`x NOT IN (подзапрос)` <=> `x <> ALL (подзапрос)` (абсолютные синонимы).
:::
::::

::::{tip} На реальной практике
:class: dropdown
:open: true
:icon: true
- Для проверки на невхождение всегда пишут `NOT IN` (или `NOT EXISTS`), потому что это читается естественнее.
- А вот ключевое слово `ALL` используют только тогда, когда нужно написать `> ALL` (больше максимального) или `< ALL` (меньше минимального).

---
:::{div}
:class: text-sm
% **Резюме**:
- `x > ALL (список)` → больше максимального элемента списка.
- `x < ALL (список)` → меньше минимального элемента списка.
- `x = ANY (список)` → то же самое, что `x IN (список)`.
- `x <> ALL (список)` → то же самое, что `x NOT IN (список)`.
:::
::::

---

#### Оператор any

Как и оператор `all`, оператор `any` позволяет сравнивать значение с членами набора значений. В отличие от `all` условие, использующее оператор `any` вычисляется как истинное – как только найдется хотя бы одно выполняющееся сравнение.

В следующем примере выполняется поиск всех клиентов, чьи суммарные платежи за прокат фильмов превышают суммарные платежи всех клиентов в Боливии, Парагвае или Чили:

In [34]:
%%sql
SELECT customer_id, SUM(amount)
FROM payment
GROUP BY customer_id
HAVING SUM(amount) > ANY (
    SELECT SUM(p.amount)
    FROM payment p
        INNER JOIN customer c ON p.customer_id = c.customer_id
        INNER JOIN address a ON c.address_id = a.address_id
        INNER JOIN city ct ON a.city_id = ct.city_id
        INNER JOIN country co ON ct.country_id = co.country_id
    WHERE co.country IN ('Bolivia', 'Paraguay', 'Chile')
    GROUP BY co.country
);

6 rows affected.

,customer_id,SUM(amount)
0,137,194.61
1,144,195.58
2,148,216.54
3,178,194.61
4,459,186.62
5,526,221.55


Подзапрос возвращает стоимость проката фильмов для всех клиентов в Боливии, Парагвае и Чили. А содержащий запрос возвращает всех клиентов, которые израсходовали сумму, превышающую расходы клиентов хотя бы одной из этих стран.

In [38]:
%%sql
SELECT co.country, SUM(p.amount)
FROM payment p
    INNER JOIN customer c ON p.customer_id = c.customer_id
    INNER JOIN address a ON c.address_id = a.address_id
    INNER JOIN city ct ON a.city_id = ct.city_id
    INNER JOIN country co ON ct.country_id = co.country_id
WHERE co.country IN ('Bolivia', 'Paraguay', 'Chile')
GROUP BY co.country
ORDER BY SUM(p.amount);

3 rows affected.

,country,SUM(p.amount)
0,Bolivia,183.53
1,Paraguay,275.38
2,Chile,328.29


Хотя большинство людей предпочитают использовать `in`, применение `= any` эквивалентно применению оператора `in`.

:::{tip} Операторы сравнения с `ANY` и `ALL`
:class: dropdown
:open: true

|             |                       |                                   |                                       |
| ----------- | --------------------- | --------------------------------- | ------------------------------------- |
| Конструкция | Логический эквивалент | Cмысл (что ищет)                  | Примечание                            |
| **> ANY**   | > MIN                 | Больше **минимального** элемента  | Достаточно быть больше хотя бы одного |
| **< ANY**   | < MAX                 | Меньше **максимального** элемента | Достаточно быть меньше хотя бы одного |
| **= ANY**   | **IN**                | Совпадает **хотя бы с одним**     | Полный аналог оператора IN            |
| **> ALL**   | > MAX                 | Строго больше **максимального**   | Больше абсолютно каждого элемента     |
| **< ALL**   | < MIN                 | Строго меньше **минимального**    | Меньше абсолютно каждого элемента     |
| **<> ALL**  | **NOT IN**            | Не совпадает **ни с одним**       | Полный аналог оператора NOT IN        |
:::

### Многостолбцовые подзапросы

До сих пор примеры подзапросов в этой главе возвращали один столбец и одну или несколько строк. Однаков в определенных ситуациях можно использовать подзапросы, возвращающие два или более столбцов.

Помочь показать полезность многостолбцовых подзапросов может следующий пример, в котором используется несколько подзапросов с одним столбцом:

:::{note} В каноническом стиле SQLFluff, dbt Style Guide и GitLab SQL Style Guide
:class: simple dropdown
:open: false
:icon: false
```sql
SELECT
    fa.actor_id,
    fa.film_id
FROM film_actor fa
WHERE
    fa.actor_id IN (
        SELECT actor_id
        FROM actor
        WHERE last_name = 'MONROE'
    )
    AND fa.film_id IN (
        SELECT film_id
        FROM film
        WHERE rating = 'PG'
    );
```
:::

In [39]:
%%sql
SELECT fa.actor_id, fa.film_id
FROM film_actor fa
WHERE fa.actor_id IN
    (SELECT actor_id FROM actor WHERE last_name = 'MONROE')
    AND fa.film_id IN
    (SELECT film_id FROM film WHERE rating = 'PG');

11 rows affected.

,actor_id,film_id
0,120,63
1,120,144
2,120,414
3,120,590
4,120,715
5,120,894
6,178,164
7,178,194
8,178,273
9,178,311


В этом запросе для идентификации всех участников с фамилией MONROE и всех фильмов с рейтингом PG используются два подзапроса, а затем содержащий запрос использует эту информацию для извлечения всех случаев, когда актер с этой фамилией появляется в фильме с рейтингом PG.

Однако можно объединить два подзапроса с одним столбцом в один подзапрос с несколькими столбцами и сравнивать результаты с двумя столбцами таблицы film_actor. Для этого условие фильтра должно указывать два столбца из таблицы film_actor в круглых скобках и том же порядке, что и в подзапросе:

:::{note} В каноническом стиле SQLFluff, dbt Style Guide и GitLab SQL Style Guide
:class: simple dropdown
:open: false
:icon: false
```sql
SELECT
    fa.actor_id,
    fa.film_id
FROM film_actor fa
WHERE
    (fa.actor_id, fa.film_id) IN (
        SELECT
            a.actor_id,
            f.film_id
        FROM actor a
            CROSS JOIN film f
        WHERE a.last_name = 'MONROE'
            AND f.rating = 'PG'
    );
```
:::

In [41]:
%%sql
SELECT actor_id, film_id
FROM film_actor
WHERE (actor_id, film_id) IN (
    SELECT a.actor_id, f.film_id
    FROM actor a
        CROSS JOIN film f
    WHERE a.last_name = 'MONROE'
        AND f.rating = 'PG'
);

11 rows affected.

,actor_id,film_id
0,120,63
1,120,144
2,120,414
3,120,590
4,120,715
5,120,894
6,178,164
7,178,194
8,178,273
9,178,311


Эта версия выполняет ту же функцию, что и в предыдущем примере, но с использованием одного подзапроса, который возвращает два столбца вместо двух подзапросов, каждый из которых возвращает один столбец.

Подзапрос в этой версии использует тип соединения, именуемого *перекрестным соединением* (которое будет рассмотрено в следующей главе). Основная идея – вернуть все комбинации актеров с фамилией MONROE (2) и всех фильмов с рейтингом PG (194), всего – 388 строк, 11 из которых могут быть найдены в таблице film_actor.

## Коррелированные подзапросы

Все подзапросы, показанные до сих пор, не зависели от содержащихся в них операторов. Это означает, что вы можете выполнить их автономно и проверить возвращаемые ими результаты.

*Коррелированный* же подзапрос *зависит* от содержащей его инструкции, ссылаясь на один или несколько ее столбцов.

В отличие от некоррелированного, *коррелированный* подзапрос *не* выполняется один раз перед выполнением содержащей его инструкции. Вместо этого коррелированный подзапрос выполняется по одному разу для каждой строки-кандидата (строки, которая может быть включена в окончательный результат).

Например, в следующем запросе используется коррелированный подзапрос для подсчета количества прокатов фильмов для каждого клиента, а содержащий запрос затем извлекает тех клиентов, которые взяли напрокат ровно 20 фильмов:

In [60]:
%%sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE 20 = (
    SELECT COUNT(*)
    FROM rental r
    WHERE r.customer_id = c.customer_id
);

15 rows affected.

,first_name,last_name
0,LAUREN,HUDSON
1,JEANETTE,GREENE
2,TARA,RYAN
3,WILMA,RICHARDS
4,JO,FOWLER
5,KAY,CALDWELL
6,DANIEL,CABRAL
7,ANTHONY,SCHWAB
8,TERRY,GRISSOM
9,LUIS,YANEZ


Ссылка на *c.customer_id* в самом конце подзапроса делает этот подзапрос *коррелированным*; содержащий запрос должен предоставлять значения для *c.customer_id* чтобы подзапрос мог быть выполнен.

В данном случае содержащий запрос извлекает все 599 строк из таблицы *customer* и выполняет подзапрос по одному разу для каждого клиента, передавая при каждом выполнении соответствующий идентификатор клиента. Если подзапрос возвращает значение 20, условие фильтра выполняется и строка добавляется к результирующему набору.

:::{div}
:class: text-sm
Работает как **цикл**: берет строку клиента → лезет в таблицу аренд → считает количество → проверяет, равно ли оно 20.
:::

:::{warning} Предупреждение
Поскольку коррелированный подзапрос будет выполняться по одному разу для каждой строки содержащего запроса, использование коррелированных подзапросов может привести к проблемам с производительностью, особенно если содержащий запрос возвращает большое количество строк.
:::

:::{topic} *Личная тренировка*
Для понимания логики решил переписать запрос через агрегацию. А выяснилось, что подход через (`JOIN` + `GROUP BY` + `HAVING`) на больших объемах данных обычно работает значительно быстрее, потому что базе не нужно запускать подзапрос отдельно для каждой строки внешнего запроса.

```sql
SELECT
    c.first_name,
    c.last_name
FROM customer c
    INNER JOIN rental r ON c.customer_id = r.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name
HAVING COUNT(*) = 20;
```
:::

Помимо условий равенства, можно использовать коррелированные подзапросы в условиях других типов, таких, например, как условие диапазона:

:::{note} В каноническом стиле SQLFluff, dbt Style Guide и GitLab SQL Style Guide
:class: simple dropdown
:open: false
:icon: false
```sql
-- Канонический (по правилам SQLFluff / dbt)
SELECT
    c.first_name,
    c.last_name
FROM customer c
WHERE
    (
        SELECT SUM(p.amount)
        FROM payment p
        WHERE p.customer_id = c.customer_id
    ) BETWEEN 180 AND 240;
```
```sql
-- Компактный (без лишнего отступа для открывающей скобки)
SELECT
    c.first_name,
    c.last_name
FROM customer c
WHERE (
    SELECT SUM(p.amount)
    FROM payment p
    WHERE p.customer_id = c.customer_id
) BETWEEN 180 AND 240;
```
:::

In [64]:
%%sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE
    (SELECT SUM(p.amount) FROM payment p
    WHERE p.customer_id = c.customer_id)
    BETWEEN 180 AND 240;

6 rows affected.

,first_name,last_name
0,RHONDA,KENNEDY
1,CLARA,SHAW
2,ELEANOR,HUNT
3,MARION,SNYDER
4,TOMMY,COLLAZO
5,KARL,SEAL


Этот вариант предыдущего запроса находит всех клиентов, чьи общие платежи за все прокаты фильмов составляют от 180 до 240 долларов. И вновь коррелированный подзапрос здесь используется 599 раз (по одному разу для каждой строки клиента); каждое выполнение подзапроса возвращает общий баланс для данного клиента.

:::{div}
:class: text-sm
Та же **циклическая механика** (*коррелированный* подзапрос), только вместо простого подсчёта строк происходит агрегация денег через SUM с последующей проверкой диапазона: берёт клиента → лезет в payment за его платежами → считает общую сумму SUM(amount) → проверяет BETWEEN 180 AND 240 → если TRUE, выводит имя и фамилию.
:::

:::{card}
Еще одно тонкое отличие показанного запроса заключается в том, что этот подзапрос находится в левой части условия (что может выглядеть немного странно, но совершенно корректно).
:::

### Оператор exist

Хотя вы часто будете встречаться с коррелированными подзапросами, используемыми в условиях равенства и диапазона, наиболее распространенный оператор, используемый для создания условий с коррелированными подзапросами – это оператор `exists`.

Оператор `exists` используется когда нужно определить существование связи безотносительно к количеству. Например, следующий запрос находит всех клиентов, которые взяли напрокат хотя бы один фильм до 25 мая 2005 года, без учета того сколько фильмов было взято:

In [65]:
%%sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE EXISTS (
    SELECT 1
    FROM rental r
    WHERE r.customer_id = c.customer_id
        AND DATE(r.rental_date) < DATE '2005-05-25'
);

8 rows affected.

,first_name,last_name
0,CHARLOTTE,HUNTER
1,DELORES,HANSEN
2,MINNIE,ROMERO
3,CASSANDRA,WALTERS
4,ANDREW,PURDY
5,MANUEL,MURRELL
6,TOMMY,COLLAZO
7,NELSON,CHRISTENSON


Используя оператор `exists` подзапрос может возвращать нуль, одну или несколько строк. И условие просто проверяет вернул ли подзапрос хотя бы одну строку.

Если посмотреть на предложение `select` подзапроса, то видно что он состоит из единственного литерала (1), так как условию в содержащем запросе нужно только знать сколько строк было возвращено. Фактические данные, возвращаемые подзапросом, значения не имеют.

::::{topic} *Личная дотошность*
:::{div}
:class: text-sm
Если выполнить подзапрос отдельно, сервер сделает следующее:
- Пойдет в таблицу rental.
- Найдет все строки, где дата аренды меньше указанной (таких строк 8 штук).
- Вместо реальных данных из таблицы (вроде rental_id, customer_id, rental_date) для каждой найденной строки вернет число 1.

Результатом будет столбец из восьми единиц:
::::

In [67]:
%%sql
SELECT 1
FROM rental r
WHERE DATE(r.rental_date) < DATE '2005-05-25';

8 rows affected.

,1
0,1
1,1
2,1
3,1
4,1
5,1
6,1
7,1


Вообще подзапрос может возвращать все что угодно:

In [66]:
%%sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE EXISTS (
    SELECT r.rental_date, r.customer_id, 'ABCD' str, 2*3/7 nmbr
    FROM rental r
    WHERE r.customer_id = c.customer_id
        AND DATE(r.rental_date) < DATE '2005-05-25'
);

8 rows affected.

,first_name,last_name
0,CHARLOTTE,HUNTER
1,DELORES,HANSEN
2,MINNIE,ROMERO
3,CASSANDRA,WALTERS
4,ANDREW,PURDY
5,MANUEL,MURRELL
6,TOMMY,COLLAZO
7,NELSON,CHRISTENSON


::::{note} Однако при использовании `EXISTS` принято указывать `SELECT 1`
:class: dropdown simple
:open: false
:icon: false
Оператору `EXISTS` абсолютно всё равно, **какие** данные возвращает подзапрос. Его интересует только один-единственный факт:
- **Вернулась хотя бы одна строка или результат пустой?** (Да/Нет, True/False).

---
:::{div}
:class: text-sm
Поэтому внутри `EXISTS` программисты пишут `SELECT 1` как символ:
- *Мне не нужны настоящие данные из таблицы (имена, даты, ID), не трать ресурсы на их извлечение – мне просто нужно знать, что строка существует!*

`SELECT 1` – это общепринятая в индустрии конвенция для подзапросов с `EXISTS`. Она визуально сигнализирует человеку, читающему код:
- *Здесь проверяется только факт наличия строки, сами значения нам не важны*.

:::
::::

Вы также можете использовать `not exists` для отбора подзапросов, которые не возвращают строк:

In [69]:
%%sql
SELECT a.first_name, a.last_name
FROM actor a
WHERE NOT EXISTS (
    SELECT 1
    FROM film_actor fa
        INNER JOIN film f ON fa.film_id = f.film_id
    WHERE fa.actor_id = a.actor_id
        AND f.rating = 'R'
);

1 rows affected.

,first_name,last_name
0,JANE,JACKMAN


Этот запрос находит всех актеров, которые никогда не снимались в фильмах с рейтингом R.

::::{note} Условие `WHERE fa.actor_id = a.actor_id` – главный инсайт главы
:class: dropdown simple
:open: false
:icon: false

:::{div}
:class: text-sm
Это мост (корреляция) между внешним миром и подзапросом, который превращает глобальный вопрос в персональный:
- Не *Есть ли вообще фильмы с рейтингом R?*
- А *Есть ли фильмы с рейтингом R именно у этого конкретного актера (у которого fa.actor_id совпадает с a.actor_id)?*

---
Условие `WHERE fa.actor_id = a.actor_id` связывает подзапрос с текущей строкой внешнего запроса, делая проверку **индивидуальной** для каждого человека. Без этого условия подзапрос проверял бы всю таблицу целиком, а не конкретного актера.
:::
::::

---

### Работа с данными с помощью коррелированных подзапросов

Все приведенные до сих пор в главе примеры были инструкциями `select`. Но не думайте, что это означает, что подзапросы бесполезны в других инструкциях SQL.

Подзапросы также широко используются в инструкциях `update`, `delete` и `insert`. Причем особенно часто коррелированные подзапросы появляются в инструкциях `update` и `delete`.

Вот пример коррелированного подзапроса, используемого для изменения столбца last_update в таблице customer:
```sql
UPDATE customer c
SET c.last_update = (
    SELECT MAX(r.rental_date)
    FROM rental r
    WHERE r.customer_id = c.customer_id
);
```
Эта инструкция изменяет каждую строку в таблице клиентов (поскольку в ней нет предложения `where`), находя последнюю дату проката для каждого клиента в таблице rental.

Хотя кажется разумным ожидать, что у каждого клиента будет хотя бы один прокат фильма, все же лучше всего проверить это, прежде чем пытаться обновить столбец last_update; в противном случае для столбца будет установлено значение `NULL`, поскольку подзапрос не вернет никакой строки.

Вот скорректированная версия инструкции `update`, на этот раз использующая предложение `where` со вторым коррелированным подзапросом:
```sql
UPDATE customer c
SET c.last_update =
    (SELECT MAX(r.rental_date) FROM rental r
    WHERE r.customer_id = c.customer_id)
WHERE EXISTS
    (SELECT 1 FROM rental r
    WHERE r.customer_id = c.customer_id);
```
Эти два коррелированных подзапроса идентичны, за исключением предложений `select`. Однако подзапрос в предложении `set` выполняется, только если условие в предложении `where` инструкции `update` истинно (т.е. если для клиента был найден хотя бы один прокат), тем самым защищая данные в столбце last_update от перезаписывания значением `NULL`.

---

Коррелированные подзапросы также распространены в инструкциях `delete`. Например, вы можете запускать сценарий обслуживания данных в конце каждого месяца, который удаляет ненужные данные. Сценарий может включать следующую инструкцию, которая удаляет те строки из таблицы customer, для которых в прошлом году не было проката фильмов:
```sql
DELETE FROM customer
WHERE 365 < ALL (
    SELECT DATEDIFF(NOW(), r.rental_date) AS days_since_last_rental
    FROM rental r
    WHERE r.customer_id = customer.customer_id
);
```

:::{note} Примечание
При использовании коррелированных запросов с инструкциями `delete` в MySQL имейте ввиду, что по какой-то причине псевдонимы таблиц при использовании `delete` не разрешены – вот почему пришлось использовать полное имя таблицы в подзапросе.

В большинстве других серверов баз данных можно указать псевдоним для таблицы customer:
```sql
DELETE FROM customer c
WHERE 365 < ALL (
    SELECT DATEDIFF(NOW(), r.rental_date) AS days_since_last_rental
    FROM rental r
    WHERE r.customer_id = c.customer_id
);
```
:::

## Применение подзапросов

Теперь, когда мы узнали о различных типах подзапросов и различных операторах, которые можно использовать для взаимодействия с данными, возвращаемыми подзапросами, пришло время изучить множество способов использования подзапросов для создания мощных инструкций SQL.

В следующих трех разделах показано как можно использовать подзапросы для создания пользовательских таблиц, построения условия и генерации значений столбцов в результирующих наборах.

### Подзапросы как источники данных

Еще в главе 3 было указано, что предложение `from` инструкции `select` содержит *таблицы*, которые будут использоваться запросом. Поскольку подзапрос генерирует результирующий набор, содержащий строки и столбцы данных, вполне допустимо включать подзапросы в предложение `from` вместе с таблицами.

Хотя на первый взгляд это может показаться интересной возможностью без особой практической ценности, использование подзапросов вместе с таблицами является одним из самых мощных инструментов, доступных при написании запросов. Вот простой пример:

In [2]:
%%sql
SELECT
    c.first_name,
    c.last_name,
    pymnt.num_rentals,
    pymnt.tot_payments
FROM customer c
    INNER JOIN (
        SELECT
            customer_id,
            COUNT(*) AS num_rentals,
            SUM(amount) AS tot_payments
        FROM payment
        GROUP BY customer_id
    ) AS pymnt
        ON c.customer_id = pymnt.customer_id;

599 rows affected.

,first_name,last_name,num_rentals,tot_payments
0,MARY,SMITH,32,118.68
1,PATRICIA,JOHNSON,27,128.73
2,LINDA,WILLIAMS,26,135.74
3,BARBARA,JONES,22,81.78
4,ELIZABETH,BROWN,38,144.62
...,...,...,...,...
594,TERRENCE,GUNDERSON,30,117.70
595,ENRIQUE,FORSYTHE,28,96.72
596,FREDDIE,DUGGAN,25,99.75
597,WADE,DELVALLE,22,83.78


В этом примере подзапрос генерирует список идентификаторов клиентов вместе с количеством прокатов фильмов и общими платежами.

Вот как выглядит результирующий набор, сгенерированный подзапросом:

In [3]:
%%sql
SELECT
    customer_id,
    COUNT(*) AS num_rentals,
    SUM(amount) AS tot_payments
FROM payment
GROUP BY customer_id;

599 rows affected.

,customer_id,num_rentals,tot_payments
0,1,32,118.68
1,2,27,128.73
2,3,26,135.74
3,4,22,81.78
4,5,38,144.62
...,...,...,...
594,595,30,117.70
595,596,28,96.72
596,597,25,99.75
597,598,22,83.78


Подзапрос получает имя pymnt и соединяется с таблицей customer через столбец customer_id. Затем содержащий запрос извлекает имя клиента из таблицы customer вместе со сводными столбцами из подзапроса pymnt.

Подзапросы, используемые в предложении `from` должны быть *некоррелированными*[^1]; они выполняются первыми и их данные хранятся в памяти до тех пор, пока не завершится выполнение содержащего запроса.

При написании запросов подзапросы предлагают огромную гибкость, потому что вы можете выйти далеко за рамки имеющегося множества доступных таблиц для создания практически любого требуемого представления данных с последующим соединением результатов с другими таблицами или подзапросами.

При написании отчетов или генерации потоков данных во внешние системы можно с помощью одного запроса решать задачи, которые иначе требовали бы вополнения нескольких запросов или применения процедурного языка программирования.

[^1]: Фактически в зависимости от используемого сервера базы данных можно включать в предложение `from` коррелированные подзапросы с помощью конструкций `cross apply` или `external apply`, но эти возможности выходят за рамки данной книги.

---

#### Создание данных

Наряду с использованием подзапросов для подытоживания существующих данных можно использовать подзапросы для генерации данных, которых в базе данных нет ни в какой форме.

Например, можно сгруппировать клиентов по сумме денег, потраченной на прокат фильмов, но при этом вы хотите использовать определения групп, которых нет в вашей базе данных. Например, допустим, что вы хотите распределить клиентов по группам:

| Группа        | Нижняя граница, долл | Верхняя граница, долл |
| ------------- | -------------------- | --------------------- |
| Small Fry     | 0                    | 74.99                 |
| Average Joes  | 75                   | 149.99                |
| Heavy Hitters | 150                  | 9999999.99            |

Чтобы сгенерировать эти группы в рамках одного запроса, требуется способ определить эти три группы. Первым шагом является создание запроса, который генерирует определения групп:

In [4]:
%%sql
SELECT 'Small Fry' name, 0 low_limit, 74.99 high_limit
UNION ALL
SELECT 'Average Joes' name, 75 low_limit, 149.99 high_limit
UNION ALL
SELECT 'Heavy Hitters' name, 150 low_limit, 9999999.99 high_limit;

3 rows affected.

,name,low_limit,high_limit
0,Small Fry,0,74.99
1,Average Joes,75,149.99
2,Heavy Hitters,150,9999999.99


Здесь использован оператор `union all`, чтобы объединить результаты трех отдельных запросов в единый результирующий набор. Каждый запрос извлекает три литерала, а результаты трех запросов объединяются для создания результирующего набора с тремя строками и тремя столбцами.

Теперь, когда у нас есть запрос для создания требуемых групп, его можно поместить в редложение `from` другого запроса для генерации групп клиентов:

In [25]:
%%sql
SELECT
    pymnt_grps.name,
    COUNT(*) num_customers
FROM (
    SELECT
        customer_id,
        COUNT(*) AS num_rentals,
        SUM(amount) AS tot_payments
    FROM payment
    GROUP BY customer_id
) AS pymnt
    INNER JOIN (
        SELECT 'Small Fry' AS name, 0 AS low_limit, 74.99 AS high_limit -- *
        UNION ALL
        SELECT 'Average Joes', 75, 149.99
        UNION ALL
        SELECT 'Heavy Hitters', 150, 9999999.99
        ) AS pymnt_grps
            ON pymnt.tot_payments
                BETWEEN pymnt_grps.low_limit
                AND pymnt_grps.high_limit
GROUP BY pymnt_grps.name;

/*  В SQL имена колонок для всего блока UNION задаются только в первом SELECT. 
    Повторять алиасы AS name, AS low_limit во второй и третьей строках
    не имеет смысла – СУБД их всё равно проигнорирует. */

3 rows affected.

,name,num_customers
0,Average Joes,515
1,Heavy Hitters,46
2,Small Fry,38


Предложение `from` содержит два подзапроса; первый подзапрос с именем pymnt (как увидели в предыдущем разделе) возвращает общее количество прокатов фильмов и общие платежи для каждого клиента, в то время как второй подзапрос с именем pymnt_grps генерирует три группы клиентов.

Два подзапроса объединяются путем определения, к какой из трех групп принадлежит каждый покупатель, а затем строки группируются по имени группы для подсчета количества клиентов в каждой группе.

:::{div}
:class: text-xs
Конечно, вы можете просто создать постоянную (или вр*е*менную) таблицу для хранения определений групп вместо использования подзапроса. Используя такой подход, вы обнаружите, что через некоторое время ваша база данных будет завалена небольшими таблицами специального назначения и вы не сможете вспомнить причину по которой было создано большинство из них.
:::

Однако, используя подзапросы, вы сможете придерживаться политики, согласно которой таблицы добавляются в базу данных только тогда, когда существует явная бизнес-потребность в хранении новых данных.

---

#### Подзапросы, ориентированные на задачу

Допустим, вы хотите создать отчет в котором будут указаны имя каждого клиента, а также его город, общее количество прокатов и общая сумма платежа. Это можно сделать соединив таблицы payment, customer, address и city, а затем сгруппировав их по имени и фамилии клиента:

In [21]:
%%sql
SELECT
    c.first_name,
    c.last_name,
    ct.city,
    SUM(p.amount) AS tot_payments,
    COUNT(*) AS tot_rentals
FROM payment p
    INNER JOIN customer c ON p.customer_id = c.customer_id
    INNER JOIN address a ON c.address_id = a.address_id
    INNER JOIN city ct ON a.city_id = ct.city_id
GROUP BY
    c.first_name,
    c.last_name,
    ct.city;

599 rows affected.

,first_name,last_name,city,tot_payments,tot_rentals
0,MARY,SMITH,Sasebo,118.68,32
1,PATRICIA,JOHNSON,San Bernardino,128.73,27
2,LINDA,WILLIAMS,Athenai,135.74,26
3,BARBARA,JONES,Myingyan,81.78,22
4,ELIZABETH,BROWN,Nantou,144.62,38
...,...,...,...,...,...
594,TERRENCE,GUNDERSON,Jinzhou,117.70,30
595,ENRIQUE,FORSYTHE,Patras,96.72,28
596,FREDDIE,DUGGAN,Sullana,99.75,25
597,WADE,DELVALLE,Lausanne,83.78,22


Этот запрос возвращает желаемые данные, но если внимательно на него посмотрите, то увидите, что таблицы customer, address и city нужны только для отображения и что в таблице payment есть все необходимое для создания группировок (customer_id и amount).

Таким образом, вы можете выделить задачу создания групп в подзапрос, а затем (для достижения желаемого конечного результата) присоединить остальные три таблицы к таблице, сгенерированной подзапросом.

Вот какой вид имеет подзапрос группировки:

In [7]:
%%sql
SELECT
    customer_id,
    COUNT(*) AS tot_rentals,
    SUM(amount) AS tot_payments
FROM payment
GROUP BY customer_id;

599 rows affected.

,customer_id,tot_rentals,tot_payments
0,1,32,118.68
1,2,27,128.73
2,3,26,135.74
3,4,22,81.78
4,5,38,144.62
...,...,...,...
594,595,30,117.70
595,596,28,96.72
596,597,25,99.75
597,598,22,83.78


Это центральная часть запроса; прочие таблицы нужны только для предоставления значимых строк вместо значения customer_id.

Следующий запрос соединяет предыдущий набор данных с тремя другими таблицами:

In [17]:
%%sql
SELECT
    c.first_name,
    c.last_name,
    ct.city,
    pymnt.tot_payments,
    pymnt.tot_rentals
FROM (
    SELECT
        customer_id,
        COUNT(*) AS tot_rentals,
        SUM(amount) AS tot_payments
    FROM payment
    GROUP BY customer_id
) AS pymnt
    INNER JOIN customer c ON pymnt.customer_id = c.customer_id
    INNER JOIN address a ON c.address_id = a.address_id
    INNER JOIN city ct ON a.city_id = ct.city_id
ORDER BY c.customer_id;

599 rows affected.

,first_name,last_name,city,tot_payments,tot_rentals
0,MARY,SMITH,Sasebo,118.68,32
1,PATRICIA,JOHNSON,San Bernardino,128.73,27
2,LINDA,WILLIAMS,Athenai,135.74,26
3,BARBARA,JONES,Myingyan,81.78,22
4,ELIZABETH,BROWN,Nantou,144.62,38
...,...,...,...,...,...
594,TERRENCE,GUNDERSON,Jinzhou,117.70,30
595,ENRIQUE,FORSYTHE,Patras,96.72,28
596,FREDDIE,DUGGAN,Sullana,99.75,25
597,WADE,DELVALLE,Lausanne,83.78,22


Я понимаю, что *красота – в глазах смотрящего*, но считаю, что эта версия запроса гораздо более привлекательна, чем большая *плоская* версия.

Этот запрос, кроме того, может выполняться быстрее, поскольку группировка выполняется по одному числовому столбцу customer_id, а не по нескольким столбцам с длинными строками customer.first_name, customer.last_name, city\.city).

---

#### Обобщенные табличные выражения

Обобщенные табличные выражения (Common table expressions, CTE), которые появились в MySQL в версии 8.0, уже были доступны на других серверах баз данных в течение некоторого времени.

CTE – это именованный подзапрос, который появляется в верхней части запроса в предложении `with`, которое может содержать несколько CTE через запятую. Помимо того что запросы при этом становятся более понятными, это также позволяет каждому CTE обращаться к любому другому CTE, определенному над ним в том же предложении `with`.

Следующий пример включает три обобщенных табличных выражения, причем второе ссылается на первое, а третье – на второе:

In [23]:
%%sql
WITH actors_s AS (
    SELECT actor_id, first_name, last_name
    FROM actor
    WHERE last_name LIKE 'S%'
),

actors_s_pg AS (
    SELECT
        s.actor_id,
        s.first_name,
        s.last_name,
        f.film_id,
        f.title
    FROM actors_s s
        INNER JOIN film_actor fa ON s.actor_id = fa.actor_id
        INNER JOIN film f ON fa.film_id = f.film_id
    WHERE f.rating = 'PG'
),

actors_s_pg_revenue AS (
    SELECT
        spg.first_name,
        spg.last_name,
        p.amount
    FROM actors_s_pg spg
        INNER JOIN inventory i ON spg.film_id = i.film_id
        INNER JOIN rental r ON i.inventory_id = r.inventory_id
        INNER JOIN payment p ON r.rental_id = p.rental_id
)

SELECT
    spg_rev.first_name,
    spg_rev.last_name,
    SUM(spg_rev.amount) AS tot_revenue
FROM actors_s_pg_revenue spg_rev
GROUP BY
    spg_rev.first_name,
    spg_rev.last_name
ORDER BY tot_revenue DESC;

9 rows affected.

,first_name,last_name,tot_revenue
0,NICK,STALLONE,692.21
1,JEFF,SILVERSTONE,652.35
2,DAN,STREEP,509.02
3,GROUCHO,SINATRA,457.97
4,SISSY,SOBIESKI,379.03
5,JAYNE,SILVERSTONE,372.18
6,CAMERON,STREEP,361.00
7,JOHN,SUVARI,296.36
8,JOE,SWANK,177.52


Этот запрос вычисляет общий доход от проката тех фильмов с рейтингом PG, актерский состав которых включает актера, фамилия которого начинается на S.

- Первый подзапрос `actors_s` находит всех актеров, чьи фамилии начинаются с S;
- Второй подзапрос `actors_s_pg` соединяет этот набор данных с таблицей film и фильтрует их фильмы с рейтингом PG;
- Третий подзапрос `actors_s_pg_revenue` соединяет этот набор данных с таблицей payment, чтобы узнать суммы, уплаченные за арену любого из этих фильмов;
- Последний запрос просто группирует данные из actors_s_pg_reverue по имени/фамилии и суммирует доходы.

Все, кто склонны использовать вр*е*менные таблицы для хранения результатов запросов для их использования в последующих запросах, могут счесть CTE привлекательной альтернативой.

:::{topic} *Личная тренировка*
Перепишем запрос из раздела *Создание данных* через `WITH` (CTE):
:::

In [24]:
%%sql
WITH pymnt AS (
    SELECT
        customer_id,
        COUNT(*) AS num_rentals,
        SUM(amount) AS tot_payments
    FROM payment
    GROUP BY customer_id
),

pymnt_grps AS (
    SELECT 'Small Fry' AS name, 0 AS low_limit, 74.99 AS high_limit
    UNION ALL
    SELECT 'Average Joes', 75, 149.99
    UNION ALL
    SELECT 'Heavy Hitters', 150, 9999999.99
)

SELECT
    pg.name,
    COUNT(*) AS num_customers
FROM pymnt p
    INNER JOIN pymnt_grps pg
        ON p.tot_payments BETWEEN pg.low_limit AND pg.high_limit
GROUP BY pg.name;

3 rows affected.

,name,num_customers
0,Average Joes,515
1,Heavy Hitters,46
2,Small Fry,38
